# Level 13: Netzwerk

Mit dem Netzwerk können wir mit anderen Computern reden. Aber wie funktioniert das?

Wir könnten da jetzt furchtbar weit ausholen, aber in der Kurzfassung: [TCP](https://de.wikipedia.org/wiki/TCP)-Sockets ermöglichen es uns, Bytes zu Programmen auf anderen (oder auch dem selben) Computer zu schicken (und zu empfangen).
In einigen von den Beispielen verwenden wir `localhost` als Ziel – das ist der aktuelle Computer.

## Hallo, Echo!

[tcpbin.com](https://tcpbin.com/) betreibt einen Echo-Server, also ein Dienst, der uns einfach das zurückschickt, was wir hinschicken.

In [6]:
from socket import create_connection

c = create_connection(("tcpbin.com", 4242))
c.send("Hallo, Welt!\n".encode())
print(c.recv(1024).decode())  # wie viele Bytes lesen wir auf einmal?
c.close()

Hallo, Welt!



Wir können uns so einen Server (mit deutlich weniger Funktionen) auch schnell selbst schreiben:
Wir können im Terminal mit `telnet localhost 4321` damit reden. (Unter Windows muss der Telnet-Client noch als optionales Windows-Feature nachinstalliert werden.)

In [15]:
from socket import AF_INET6, create_server

s = create_server(("::", 4321), family=AF_INET6, dualstack_ipv6=True)
conn, client = s.accept()  # ein echter Server würde in einer Schleife mehrere Verbindungen annehmen
data = conn.recv(1024).decode()
print(f"Received {data!r} from {client}")
conn.send(data.encode())  # ein echter Server würde in einer Schleife mehrere Zeilen beantworten
conn.close()
s.close()

Received 'Das ist ein Text.\r\n' from ('::1', 58430, 0, 0)


## Hyper Hyper

Jetzt, wo wir Daten übers Netzwerk schicken können, schauen wir uns [HTTP](https://de.wikipedia.org/wiki/HTTP) an, eines der am meisten genutzten Protokolle.

In [16]:
from socket import create_connection

c = create_connection(("example.net", 80))
c.send("GET / HTTP/1.1\n".encode())
c.send("Host: example.net\n\n".encode())
print(c.recv(1024).decode())  # wie viele Bytes lesen wir auf einmal?
c.close()

HTTP/1.1 200 OK
Date: Sat, 12 Sep 2026 14:09:08 GMT
Content-Type: text/html
Transfer-Encoding: chunked
Connection: keep-alive
Server: cloudflare
Last-Modified: Fri, 11 Sep 2026 17:42:00 GMT
Allow: GET, HEAD
Accept-Ranges: bytes
Age: 8668
Cache-Control: max-age=14400
cf-cache-status: HIT
CF-RAY: a39f7cfbf955d396-FRA

22f
<!doctype html><html lang="en"><head><title>Example Domain</title><link rel="icon" href="data:,"><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>

0




Wir haben hier gerade die Seite [example.net](https://example.net) aufgerufen und dabei können wir verschiedene Dinge sehen:

1. Anfrage und Antwort sind ähnlich aufgebaut
2. die erste Zeile ist jeweils die wichtigste:
   * sie enthält immer `HTTP` und die Version (`1.1`).
   * bei der Anfrage enthält sie zusätzlich noch das Verb (`GET`) und den Pfad (`/`)
   * bei der Antwort enthält sie zusätzlich noch den Statuscode (`200 OK`)
3. danach kommen die Header, jeweils auf einer Zeile (`Key: Value`)
4. danach kommt eine Leerzeile
5. danach kommt der Inhalt (unsere Anfrage hat keinen)
6. wir enden mit einer Leerzeile

## das Ökosystem

Wir könnten jetzt selbst HTTP sprechen über rohe Sockets, aber das ist (wie eben gesehen) doch etwas aufwändig.
Stattdessen gibt es fertige Software, die uns die Grundlagen des Protokolls abnimmt.

Wir fangen an mit `requests` als HTTP-Client:

In [18]:
import requests

req = requests.get("http://example.net")
print(req.status_code)
print(req.text)

200
<!doctype html><html lang="en"><head><title>Example Domain</title><link rel="icon" href="data:,"><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more</a></p></div></body></html>



Für Webserver in Python ist der Standard **WSGI**, das Web Server Gateway Interface:
Das Konzept sieht vor, dass wir eine Funktion definieren müssen, die als Parameter die Anfrage bekommt und die Antwort zurückgibt.
Wir können WSGI-Anwendungen u.a. mit [gunicorn](https://gunicorn.org/) oder [waitress](https://docs.pylonsproject.org/projects/waitress/en/stable/) ausführen:
`gunicorn myfile:app`

In [19]:
from typing import Any, Callable, Iterable

def app(environ: dict[str, Any], start_response: Callable) -> Iterable[bytes]:
    start_response("200 OK", [
        ("Content-Type", "text/plain"),
    ])
    return [b"Hallo, Welt!"]

Das Interface ist zwar sehr flexibel, aber eher umständlich: Wir müssen z.B. Query-Parameter und Formular-Inhalte selbst parsen. Stattdessen können wir ein WSGI-Framework wie [Flask](https://flask.palletsprojects.com/) verwenden:

In [21]:
# taken from the Flask documentation
from flask import Flask

app = Flask(__name__)


@app.route("/")
def hallo():
    return "Hallo, Welt!"